[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week02_backprop/week02_master_capstone.ipynb)

# Week 2 Master Capstone - From Manual Backprop to `nn.Module`

## 1. Scope / objectives

Purpose: make the grader's path explicit from hand-derived backprop to a reproducible PyTorch submission; verify: each stage has a focused smoke test, parity check, or checkpoint check; Week 2: this is the bridge from calculus to a trainable model.

- NumPy reference math and finite-difference gradient check
- Torch parity without autograd
- Autograd comparison, `nn.Module` training, checkpointing, and diagnostics


In [ ]:
# Install torch if running in an environment where it is not pre-installed (e.g. bare Colab).
try:
    import torch  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
    import torch  # noqa: F401

## 2. Reproducibility / config

Purpose: lock the seed, constants, logging, and checkpoint path used by every later cell; verify: the run is seeded and `CKPT_PATH` resolves in the notebook working directory, so `week02_best_two_layer_xor.pt` is written relative to the notebook kernel's current working directory during execution; Week 2: this keeps the manual math and Torch parity checks comparable.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("week02")

SEED = 42
N_SAMPLES = 500
D = 2
H = 4
OUT = 1
BATCH_SIZE = 16
LR = 0.1
EPOCHS = 200
VAL_FRACTION = 0.2
CKPT_PATH = (
    Path(__file__).parent / "week02_best_two_layer_xor.pt"
    if "__file__" in dir()
    else Path("week02_best_two_layer_xor.pt")
)

In [ ]:
def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything(SEED)

## 3. Toy XOR data

Purpose: define and visualize the fixed XOR problem used throughout the notebook; verify: the label rule is `1[x1*x2 < 0]` and the class balance looks sane; Week 2: this is the smallest nonlinear dataset that still needs backprop.


In [ ]:
def generate_xor_data(n_samples=N_SAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.0, 1.0, size=(n_samples, 2))
    y = (X[:, 0] * X[:, 1] < 0).astype(np.float64)
    return X, y


X_np, y_np = generate_xor_data(N_SAMPLES, seed=SEED)
print("X shape:", X_np.shape, "y shape:", y_np.shape)
print("class balance (mean of y):", float(y_np.mean()))

fig, ax = plt.subplots(figsize=(3.2, 3.2))
ax.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], s=10, alpha=0.55, label="y=0", color="tab:blue")
ax.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], s=10, alpha=0.55, label="y=1", color="tab:orange")
ax.set_title("Toy XOR data")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend(frameon=False, fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()

## 4. Manual NumPy forward/backward

Purpose: implement the one-hidden-layer MLP and its analytic gradient in NumPy; verify: the parameter shapes match the canonical `(H, D)` / `(1, H)` layout; Week 2: this is the reference math before Torch enters the picture.


In [ ]:
def init_numpy_params(seed=SEED + 1, d=D, h=H, out=OUT):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0.0, 0.1, size=(h, d))
    b1 = np.zeros((h,))
    W2 = rng.normal(0.0, 0.1, size=(out, h))
    b2 = np.zeros((out,))
    return W1, b1, W2, b2


def relu_np(u):
    return np.maximum(0.0, u)


def relu_prime_np(u):
    return (u > 0.0).astype(u.dtype)


def forward_numpy(x, W1, b1, W2, b2):
    a1 = W1 @ x + b1
    h1 = relu_np(a1)
    f = (W2 @ h1 + b2).reshape(-1)[0]
    return a1, h1, f


def loss_numpy(f, y):
    return 0.5 * (f - y) ** 2


def backward_numpy(x, y, a1, h1, f, W2):
    df = f - y
    dW2 = df * h1[None, :]
    db2 = np.array([df], dtype=W2.dtype)
    dh = W2[0] * df
    da1 = dh * relu_prime_np(a1)
    dW1 = da1[:, None] @ x[None, :]
    db1 = da1
    return dW1, db1, dW2, db2


def flatten_numpy_params(W1, b1, W2, b2):
    return np.concatenate([W1.ravel(), b1.ravel(), W2.ravel(), b2.ravel()])


def unflatten_numpy_params(theta, W1_shape, b1_shape, W2_shape, b2_shape):
    n1 = int(np.prod(W1_shape))
    n2 = int(np.prod(b1_shape))
    n3 = int(np.prod(W2_shape))
    n4 = int(np.prod(b2_shape))
    i0, i1, i2, i3, i4 = 0, n1, n1 + n2, n1 + n2 + n3, n1 + n2 + n3 + n4
    W1 = theta[i0:i1].reshape(W1_shape)
    b1 = theta[i1:i2].reshape(b1_shape)
    W2 = theta[i2:i3].reshape(W2_shape)
    b2 = theta[i3:i4].reshape(b2_shape)
    return W1, b1, W2, b2

In [ ]:
W1_np, b1_np, W2_np, b2_np = init_numpy_params(seed=SEED + 1)
print("parameter shapes:", W1_np.shape, b1_np.shape, W2_np.shape, b2_np.shape)

In [ ]:
def loss_from_theta(theta, x, y, shapes):
    W1_shape, b1_shape, W2_shape, b2_shape = shapes
    W1, b1, W2, b2 = unflatten_numpy_params(theta, W1_shape, b1_shape, W2_shape, b2_shape)
    _, _, f = forward_numpy(x, W1, b1, W2, b2)
    return loss_numpy(f, y)


def analytic_grad(theta, x, y, shapes):
    W1_shape, b1_shape, W2_shape, b2_shape = shapes
    W1, b1, W2, b2 = unflatten_numpy_params(theta, W1_shape, b1_shape, W2_shape, b2_shape)
    a1, h1, f = forward_numpy(x, W1, b1, W2, b2)
    dW1, db1, dW2, db2 = backward_numpy(x, y, a1, h1, f, W2)
    return flatten_numpy_params(dW1, db1, dW2, db2)


def numeric_grad(theta, x, y, shapes, eps=1e-5):
    grad = np.zeros_like(theta)
    theta_plus = theta.copy()
    theta_minus = theta.copy()
    for i in range(len(theta)):
        theta_plus[i] = theta[i] + eps
        theta_minus[i] = theta[i] - eps
        loss_plus = loss_from_theta(theta_plus, x, y, shapes)
        loss_minus = loss_from_theta(theta_minus, x, y, shapes)
        grad[i] = (loss_plus - loss_minus) / (2 * eps)
        theta_plus[i] = theta[i]
        theta_minus[i] = theta[i]
    return grad


def pick_nonboundary_sample(X, W1, b1, W2, b2, tol=1e-4):
    for idx, x in enumerate(X):
        a1, _, _ = forward_numpy(x, W1, b1, W2, b2)
        if np.all(np.abs(a1) > tol):
            return idx
    return 0

## 5. Finite-difference gradient check

Purpose: compare the analytic NumPy gradient against a central finite-difference estimate; verify: the chosen sample avoids the ReLU boundary and the max diff stays tiny; Week 2: this is the algebra correctness gate before Torch parity.


In [ ]:
sample_idx = pick_nonboundary_sample(X_np, W1_np, b1_np, W2_np, b2_np)
x0, y0 = X_np[sample_idx], y_np[sample_idx]
a1_0, h1_0, f0 = forward_numpy(x0, W1_np, b1_np, W2_np, b2_np)
theta0 = flatten_numpy_params(W1_np, b1_np, W2_np, b2_np)
shapes = (W1_np.shape, b1_np.shape, W2_np.shape, b2_np.shape)

g_analytic = analytic_grad(theta0, x0, y0, shapes)
g_numeric = numeric_grad(theta0, x0, y0, shapes, eps=1e-5)
abs_diff = np.abs(g_analytic - g_numeric)
rel_diff = abs_diff / (np.abs(g_numeric) + 1e-12)

print(f"sample idx: {sample_idx}")
print(f"min |a1|: {np.min(np.abs(a1_0)):.2e}")
print(f"forward f: {f0:.6f} | loss: {loss_numpy(f0, y0):.6f}")
print(f"max abs diff: {abs_diff.max():.2e}")
print(f"max rel diff: {rel_diff.max():.2e}")
assert abs_diff.max() < 1e-5, "gradient check failed"

## 6. PyTorch tensor bridge without autograd

Purpose: define the Torch mirror of the NumPy MLP and verify forward and backward parity in two steps; verify: the helper functions, cloned-tensor output, and manual gradients all match NumPy; Week 2: this isolates framework mechanics from autograd.


### Helpers

Purpose: mirror the NumPy forward/backward pass in Torch; verify: the helper formulas and tensor shapes stay aligned with the reference math.


In [ ]:
def relu_torch(x):
    return torch.relu(x)


def forward_torch(x, W1, b1, W2, b2):
    a1 = x @ W1.T + b1
    h1 = relu_torch(a1)
    f = (W2 @ h1 + b2).reshape(-1)[0]
    return a1, h1, f


def backward_torch(x, y, a1, h1, f, W2):
    df = f - y
    dW2 = df * h1[None, :]
    db2 = df.reshape(1)
    dh = W2[0] * df
    da1 = dh * (a1 > 0).to(a1.dtype)
    dW1 = da1[:, None] @ x[None, :]
    db1 = da1
    return dW1, db1, dW2, db2

### Forward parity

Purpose: clone the NumPy parameters into Torch and check the one-sample forward pass; verify: the Torch activations and output match the NumPy reference within tolerance.


In [ ]:
W1_t0 = torch.tensor(W1_np, dtype=torch.float32)
b1_t0 = torch.tensor(b1_np, dtype=torch.float32)
W2_t0 = torch.tensor(W2_np, dtype=torch.float32)
b2_t0 = torch.tensor(b2_np, dtype=torch.float32)
for tensor in [W1_t0, b1_t0, W2_t0, b2_t0]:
    tensor.requires_grad_(False)

x0_t = torch.tensor(x0, dtype=torch.float32)
y0_t = torch.tensor(y0, dtype=torch.float32)

a1_t, h1_t, f_t = forward_torch(x0_t, W1_t0, b1_t0, W2_t0, b2_t0)
print(f"|f_numpy - f_torch| = {abs(f0 - f_t.item()):.2e}")
assert np.allclose(a1_0, a1_t.detach().numpy(), atol=1e-6)
assert np.allclose(h1_0, h1_t.detach().numpy(), atol=1e-6)
assert abs(f0 - f_t.item()) < 1e-6

### Backward parity

Purpose: compare the manual Torch gradients against the NumPy reference on the same sample; verify: the per-parameter max absolute diffs stay tiny.


In [ ]:
dW1_t0, db1_t0, dW2_t0, db2_t0 = backward_torch(x0_t, y0_t, a1_t, h1_t, f_t, W2_t0)
print("manual torch gradient shapes:", dW1_t0.shape, db1_t0.shape, dW2_t0.shape, db2_t0.shape)

dW1_np0, db1_np0, dW2_np0, db2_np0 = backward_numpy(x0, y0, a1_0, h1_0, f0, W2_np)
torch_backward_dW1 = np.max(np.abs(dW1_t0.detach().numpy() - dW1_np0))
torch_backward_db1 = np.max(np.abs(db1_t0.detach().numpy() - db1_np0))
torch_backward_dW2 = np.max(np.abs(dW2_t0.detach().numpy() - dW2_np0))
torch_backward_db2 = np.max(np.abs(db2_t0.detach().numpy() - db2_np0))
torch_backward_max_abs_diff = max(
    torch_backward_dW1,
    torch_backward_db1,
    torch_backward_dW2,
    torch_backward_db2,
)
print(f"|dW1_torch - dW1_numpy|_max = {torch_backward_dW1:.2e}")
print(f"|db1_torch - db1_numpy|_max = {torch_backward_db1:.2e}")
print(f"|dW2_torch - dW2_numpy|_max = {torch_backward_dW2:.2e}")
print(f"|db2_torch - db2_numpy|_max = {torch_backward_db2:.2e}")
assert torch_backward_max_abs_diff < 1e-6, "torch and NumPy backward gradients differ"

In [ ]:
# ReLU boundary convention: torch.relu grad at x==0 should be 0, matching (a1>0)
a1_boundary = torch.tensor([0.0], requires_grad=True)
torch.relu(a1_boundary).backward(torch.ones(1))
assert a1_boundary.grad.item() == 0.0, (
    f"Expected grad 0 at boundary, got {a1_boundary.grad.item()}"
)
print(f"[OK] relu boundary grad = {a1_boundary.grad.item():.1f} (matches (a1>0)=0 convention)")

## 7. Fixed-batch autograd parity

Purpose: compare manual batch gradients against Torch autograd on a fixed slice; verify: the batch stays off the ReLU boundary and the max gradient diff is tiny; Week 2: this is the mini-batch bridge from hand-derived calculus to autograd.


In [ ]:
batch_idx = np.array([0, 1, 2, 3])
Xb_np = X_np[batch_idx].astype(np.float32)
yb_np = y_np[batch_idx].astype(np.float32)
W1_np32 = W1_np.astype(np.float32)
b1_np32 = b1_np.astype(np.float32)
W2_np32 = W2_np.astype(np.float32)
b2_np32 = b2_np.astype(np.float32)

# Confirm the fixed batch stays comfortably away from the ReLU boundary.
a1_probe = Xb_np @ W1_np32.T + b1_np32
assert np.all(np.abs(a1_probe) > 1e-4), "batch touches the ReLU boundary"
print(f"batch idx: {batch_idx.tolist()} | min |a1|: {np.min(np.abs(a1_probe)):.2e}")


def forward_torch_batch(X, W1, b1, W2, b2):
    a1 = X @ W1.T + b1
    h1 = relu_torch(a1)
    f = (h1 @ W2.T + b2).squeeze(-1)
    return a1, h1, f


W1_ag = torch.tensor(W1_np32, dtype=torch.float32, requires_grad=True)
b1_ag = torch.tensor(b1_np32, dtype=torch.float32, requires_grad=True)
W2_ag = torch.tensor(W2_np32, dtype=torch.float32, requires_grad=True)
b2_ag = torch.tensor(b2_np32, dtype=torch.float32, requires_grad=True)
params_ag = [W1_ag, b1_ag, W2_ag, b2_ag]

for p in params_ag:
    if p.grad is not None:
        p.grad.zero_()

Xb_t = torch.tensor(Xb_np, dtype=torch.float32)
yb_t = torch.tensor(yb_np, dtype=torch.float32)

a1_ag, h1_ag, f_ag = forward_torch_batch(Xb_t, W1_ag, b1_ag, W2_ag, b2_ag)
loss_ag = 0.5 * ((f_ag - yb_t) ** 2).mean()
loss_ag.backward()

g_auto = torch.cat([
    W1_ag.grad.reshape(-1),
    b1_ag.grad.reshape(-1),
    W2_ag.grad.reshape(-1),
    b2_ag.grad.reshape(-1),
]).detach().cpu().numpy()

with torch.no_grad():
    a1_np = a1_ag.detach().cpu().numpy()
    h1_np = h1_ag.detach().cpu().numpy()
    f_np = f_ag.detach().cpu().numpy()
    df = (f_np - yb_np) / len(batch_idx)
    dW2_m = df[:, None].T @ h1_np
    db2_m = np.array([df.sum()], dtype=np.float32)
    dh = df[:, None] * W2_np32[0][None, :]
    da1 = dh * relu_prime_np(a1_np)
    dW1_m = da1.T @ Xb_np
    db1_m = da1.sum(axis=0)
    g_manual = np.concatenate([
        dW1_m.reshape(-1),
        db1_m.reshape(-1),
        dW2_m.reshape(-1),
        db2_m.reshape(-1),
    ])

batch_max_abs_diff = float(np.max(np.abs(g_auto - g_manual)))
print(f"manual-vs-autograd batch max abs diff = {batch_max_abs_diff:.2e}")
assert batch_max_abs_diff < 1e-6, "fixed-batch manual and autograd gradients differ"

## 8. Dataset / train-val split

Purpose: create the held-out split and DataLoaders used for training; verify: the split is stratified and the loader sizes are as expected; Week 2: this turns the toy problem into a proper submission run.


In [ ]:
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_np,
    y_np,
    test_size=VAL_FRACTION,
    random_state=SEED,
    stratify=y_np,
)

train_ds = TensorDataset(
    torch.tensor(X_train_np, dtype=torch.float32),
    torch.tensor(y_train_np, dtype=torch.float32),
)
val_ds = TensorDataset(
    torch.tensor(X_val_np, dtype=torch.float32),
    torch.tensor(y_val_np, dtype=torch.float32),
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(val_ds, batch_size=len(val_ds), shuffle=False)

print(f"train/val sizes: {len(train_ds)} / {len(val_ds)}")
print(f"train batches per epoch: {len(train_loader)}")
print(f"val batches: {len(val_loader)}")

## 9. `nn.Module` model

Purpose: package the same MLP math into a trainable module; verify: the parameter shapes and output shape match the manual reference; Week 2: this is the object-oriented form used for training.


In [ ]:
class TwoLayerXOR(torch.nn.Module):
    def __init__(self, d=D, h=H, out=OUT, seed=SEED + 1):
        super().__init__()
        rng = np.random.default_rng(seed)
        self.W1 = torch.nn.Parameter(torch.tensor(rng.normal(0.0, 0.1, size=(h, d)), dtype=torch.float32))
        self.b1 = torch.nn.Parameter(torch.zeros(h, dtype=torch.float32))
        self.W2 = torch.nn.Parameter(torch.tensor(rng.normal(0.0, 0.1, size=(out, h)), dtype=torch.float32))
        self.b2 = torch.nn.Parameter(torch.zeros(out, dtype=torch.float32))
        self.last_h1 = None

    def forward(self, x):
        a1 = x @ self.W1.T + self.b1
        h1 = torch.relu(a1)
        self.last_h1 = h1.detach()
        return (h1 @ self.W2.T + self.b2).squeeze(-1)


model = TwoLayerXOR()
print(model)
print({name: tuple(param.shape) for name, param in model.named_parameters()})

In [ ]:
# --- Engineering: shape invariant pre-flight check ---
_x_probe = torch.zeros(1, D)
assert model(_x_probe).shape == (1,), "model output shape mismatch"
n_params = sum(p.numel() for p in model.parameters())
assert n_params == H * D + H + OUT * H + OUT, f"unexpected param count: {n_params}"
print(f"[OK] model I/O shapes correct | total trainable params: {n_params}")
del _x_probe

## 10. `nn.Sequential` parity

Purpose: show the same model can be expressed in `nn.Sequential`; verify: copied weights produce matching outputs; Week 2: this proves the module math does not depend on the custom class wrapper.


In [ ]:
import torch.nn as nn

seq_model = nn.Sequential(
    nn.Linear(D, H),
    nn.ReLU(),
    nn.Linear(H, OUT),
)

with torch.no_grad():
    seq_model[0].weight.copy_(model.W1)
    seq_model[0].bias.copy_(model.b1)
    seq_model[2].weight.copy_(model.W2)
    seq_model[2].bias.copy_(model.b2)

xb_seq = torch.tensor(X_np[:4], dtype=torch.float32)
with torch.no_grad():
    model_out = model(xb_seq)
    seq_out = seq_model(xb_seq).squeeze(-1)

seq_max_abs_diff = float((model_out - seq_out).abs().max().item())
print(seq_model)
print(f"seq parity max abs diff = {seq_max_abs_diff:.2e}")
assert seq_max_abs_diff < 1e-6, "sequential and module outputs differ"

## 11. Training loop

Purpose: run the seeded SGD loop and track validation metrics; verify: loss, accuracy, gradient norm, and hidden activity are recorded and the best checkpoint updates on validation loss; Week 2: this is the actual learning run the grader checks.


### Setup

Purpose: clear the checkpoint and initialize the loss/optimizer; verify: the training run starts from a clean state.


> **Loss convention note:** training uses `torch.nn.MSELoss(reduction="mean")`, which computes
> $\frac{1}{N}\sum(f_i - y_i)^2$ — **twice** the handout's $\frac{1}{2}(f-y)^2$ scalar.
> This rescales the effective learning rate (grad magnitude doubles relative to the ½-convention)
> but does **not** shift the optimum or affect the ½-convention parity checks above,
> which use the single-sample `0.5*(f-y)**2` loss directly.

> **Accuracy threshold note:** `evaluate` reports classification accuracy via `preds >= 0.5`.
> The XOR labels are in $\{0, 1\}$ and the linear-output head is trained with MSE to regress toward
> those targets, so `0.5` is simply the midpoint decision boundary between the two label values.
> Accuracy at that threshold is therefore an honest, meaningful metric even though the raw output
> is an unbounded regression value rather than a calibrated probability (there is no sigmoid).

In [ ]:
if CKPT_PATH.exists():
    CKPT_PATH.unlink()

loss_fn = torch.nn.MSELoss(reduction="mean")
optimizer = torch.optim.SGD(model.parameters(), lr=LR)

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0.0
    total_count = 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb)
            batch_loss = loss_fn(preds, yb)
            batch_count = len(xb)
            total_loss += batch_loss.item() * batch_count
            total_correct += ((preds >= 0.5).float() == yb).float().sum().item()
            total_count += batch_count
    return total_loss / total_count, total_correct / total_count


train_loss_hist = []
val_loss_hist = []
val_acc_hist = []
grad_norm_hist = []
relu_activity_hist = []

best_val_loss = float("inf")
best_val_acc = 0.0
best_epoch = 0

### Epoch loop

Purpose: train, validate, and save the best checkpoint; verify: the full epoch loop still logs and checkpoints exactly once per improvement.


In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_count = 0
    batch_grad_norms = []
    epoch_active = 0.0
    epoch_hidden = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()

        total_norm_sq = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total_norm_sq += p.grad.norm().item() ** 2
        batch_grad_norms.append(total_norm_sq ** 0.5)
        epoch_active += (model.last_h1 > 0).float().sum().item()
        epoch_hidden += model.last_h1.numel()

        optimizer.step()
        running_loss += loss.item() * len(xb)
        running_count += len(xb)

    mean_train_loss = running_loss / running_count
    train_loss_hist.append(mean_train_loss)
    grad_norm_hist.append(float(np.mean(batch_grad_norms)))
    relu_activity_hist.append(epoch_active / epoch_hidden)

    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    val_loss_hist.append(val_loss)
    val_acc_hist.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "state_dict": model.state_dict(),
            },
            CKPT_PATH,
        )

    if epoch == 1 or epoch % 20 == 0:
        logger.info(
            "epoch %03d | train loss %.4f | val loss %.4f | val acc %.4f",
            epoch, mean_train_loss, val_loss, val_acc,
        )

## 12. Checkpoint save/reload

Purpose: reload the best validation checkpoint and smoke-test the saved state; verify: the loaded metrics reproduce the saved validation values; Week 2: this confirms the artifact is real and restorable.


In [ ]:
assert CKPT_PATH.exists(), f"missing checkpoint: {CKPT_PATH}"

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
best_model = TwoLayerXOR()
best_model.load_state_dict(ckpt["state_dict"])
loaded_val_loss, loaded_val_acc = evaluate(best_model, val_loader, loss_fn)

print(f"checkpoint epoch: {ckpt['epoch']}")
print(f"saved  val loss/acc: {ckpt['val_loss']:.6f} / {ckpt['val_acc']:.4f}")
print(f"loaded val loss/acc: {loaded_val_loss:.6f} / {loaded_val_acc:.4f}")

assert abs(loaded_val_loss - ckpt["val_loss"]) < 1e-7
assert abs(loaded_val_acc - ckpt["val_acc"]) < 1e-7

## 13. Diagnostics

Purpose: give a quick visual scan of the run; verify: the best epoch is marked consistently across loss, accuracy, gradient norm, and ReLU activity; Week 2: this is the one-glance training health check.


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
ax = ax.ravel()

ax[0].plot(train_loss_hist, label="train", marker="o", markevery=[best_epoch - 1], ms=5, mfc="white")
ax[0].plot(val_loss_hist, label="val", marker="s", markevery=[best_epoch - 1], ms=5, mfc="white")
ax[0].axvline(best_epoch - 1, color="black", ls="--", alpha=0.3)
ax[0].set_title("Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("MSE")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(val_acc_hist, color="tab:green", marker="o", markevery=[best_epoch - 1], ms=5, mfc="white")
ax[1].axvline(best_epoch - 1, color="black", ls="--", alpha=0.3)
ax[1].set_title("Validation accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].set_ylim(0.0, 1.05)
ax[1].grid(True, alpha=0.3)

ax[2].plot(grad_norm_hist, color="tab:orange", marker="o", markevery=[best_epoch - 1], ms=5, mfc="white")
ax[2].axvline(best_epoch - 1, color="black", ls="--", alpha=0.3)
ax[2].set_title("Mean gradient norm")
ax[2].set_xlabel("Epoch")
ax[2].set_ylabel("L2 norm")
ax[2].grid(True, alpha=0.3)

ax[3].plot(relu_activity_hist, color="tab:purple", marker="o", markevery=[best_epoch - 1], ms=5, mfc="white")
ax[3].axvline(best_epoch - 1, color="black", ls="--", alpha=0.3)
ax[3].set_title("Mean hidden ReLU activity")
ax[3].set_xlabel("Epoch")
ax[3].set_ylabel("Fraction active")
ax[3].set_ylim(0.0, 1.05)
ax[3].grid(True, alpha=0.3)

plt.tight_layout()

## 14. Final interpretation

Purpose: tie the checks together and state the reviewer-facing conclusion; verify: the printed metrics below agree with the gradient checks, parity checks, training run, and checkpoint smoke test; Week 2: this closes the submission with an evidence-backed summary.


In [ ]:
print(f"gradient-check sample idx: {sample_idx}")
print(f"NumPy max abs grad diff: {abs_diff.max():.2e}")
print(f"Torch forward parity max abs diff: {abs(f0 - f_t.item()):.2e}")
print(f"Torch manual-vs-NumPy backward max abs diff: {torch_backward_max_abs_diff:.2e}")
print(f"fixed-batch manual-vs-autograd max abs diff: {batch_max_abs_diff:.2e}")
print(f"nn.Sequential parity max abs diff: {seq_max_abs_diff:.2e}")
print(f"best epoch: {best_epoch}")
print(f"best val loss: {best_val_loss:.6f}")
print(f"best val acc: {best_val_acc:.4f}")
print(f"mean ReLU activity at best epoch: {relu_activity_hist[best_epoch - 1]:.4f}")
print(f"checkpoint smoke test: passed ({loaded_val_loss:.6f} / {loaded_val_acc:.4f})")
print(f"checkpoint path: {CKPT_PATH}")